In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# Load data
df = pd.read_csv("../data/caesarian.csv")

# Split into X (features) and y (target)
X = df.drop(columns=["Caesarian"])
y = df["Caesarian"]  # already 0/1

# Train/test split (stratify keeps class ratio similar)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: scaler + logistic regression
# solver='saga' so we can represent L1 vs L2 via l1_ratio (modern / future-proof)
pipe = Pipeline([
    ("scaler", StandardScaler()),  # placeholder (GridSearch will swap it)
    ("logreg", LogisticRegression(
        solver="saga",
        max_iter=5000,
        random_state=42
    ))
])

# Hyperparameter grid:
# - scaler: StandardScaler (z-score) vs MinMaxScaler
# - C: regularization strength (inverse)
# - l1_ratio: 0.0 => L2, 1.0 => L1
param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler()],
    "logreg__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logreg__l1_ratio": [0.0, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True
)

# Train with grid search
grid.fit(X_train, y_train)

# Evaluate on test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", f"{grid.best_score_:.4f}")
print("Test Accuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
print("Test ROC-AUC:", f"{roc_auc_score(y_test, y_prob):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Best Parameters: {'logreg__C': 1, 'logreg__l1_ratio': 0.0, 'scaler': MinMaxScaler()}
Best CV Accuracy: 0.6885
Test Accuracy: 0.6250
Test ROC-AUC: 0.7937

Confusion Matrix:
 [[2 5]
 [1 8]]

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.29      0.40         7
           1       0.62      0.89      0.73         9

    accuracy                           0.62        16
   macro avg       0.64      0.59      0.56        16
weighted avg       0.64      0.62      0.58        16

